In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn pyreadstat


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import zipfile
from pathlib import Path

import pyreadstat
import pandas as pd
import numpy as pd

In [3]:
!pwd

/data/users/jupyter-yos225/venvs/proflee/drug


In [4]:
from pathlib import Path
import zipfile

def extract_selected_teds_stata_zips(base_dir, overwrite = False):
    """
    Extract only selected TEDS-A Stata zip files for multiple years.
    - TEDS-A-2021-DS0001-bndl-data-stata_v3.zip
    - TEDS-A-2022-DS0001-bndl-data-stata_v2.zip
    - teds-a-2023-ds0001-bndl-data-stata_v1.zip
    """

    base_dir = Path(base_dir)

    zip_files = {
        2021: "TEDS-A-2021-DS0001-bndl-data-stata_v3.zip",
        2022: "TEDS-A-2022-DS0001-bndl-data-stata_v2.zip",
        2023: "teds-a-2023-ds0001-bndl-data-stata_v1.zip"  
    }

    summary = {}

    for year, zip_name in zip_files.items():
        zip_path = base_dir / zip_name
        extract_dir = base_dir / f"TEDS_A_{year}_stata_extracted"

        print(f"\n====== {year} ======")

        if not zip_path.exists():
            print(f"[ERROR] Zip file not found: {zip_path}")
            summary[year] = {
                "status": "missing_zip",
                "zip_path": zip_path,
                "extract_dir": extract_dir,
                "dta_files": [],
                "do_files": [],
                "all_files": [],
            }
            continue

        if extract_dir.exists() and not overwrite:
            print(f"[INFO] Extract folder already exists, skipping extraction:")
            print(f"       {extract_dir.resolve()}")
        else:
            extract_dir.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_dir)

            print(f"[OK] Extracted to: {extract_dir.resolve()}")

        all_files = sorted([p for p in extract_dir.glob("*") if p.is_file()])

        dta_files = sorted([p for p in all_files if p.suffix.lower() == ".dta"])
        do_files = sorted([p for p in all_files if p.suffix.lower() == ".do"])

        print(f"[INFO] Total files: {len(all_files)}")
        print(f"[INFO] DTA files: {len(dta_files)}")
        for f in dta_files:
            print(" -", f)

        print(f"[INFO] DO files: {len(do_files)}")
        for f in do_files:
            print(" -", f)
        summary[year] = {
            "status": "ok",
            "zip_path": zip_path,
            "extract_dir": extract_dir,
            "dta_files": dta_files,
            "do_files": do_files,
            "all_files": all_files,
        }

    return summary

In [5]:
BASE_DIR = Path(os.environ.get("NSDUH_DATA_DIR", "."))

summary = extract_selected_teds_stata_zips(
    base_dir = BASE_DIR,
    overwrite = False
)


====== 2021 ======
[INFO] Extract folder already exists, skipping extraction:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted
[INFO] Total files: 1
[INFO] DTA files: 1
 - /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta
[INFO] DO files: 0

====== 2022 ======
[INFO] Extract folder already exists, skipping extraction:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted
[INFO] Total files: 1
[INFO] DTA files: 1
 - /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta
[INFO] DO files: 0

====== 2023 ======
[INFO] Extract folder already exists, skipping extraction:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted
[INFO] Total files: 1
[INFO] DTA files: 1
 - /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta
[INFO] DO files: 0


In [6]:
all_dta_files = []
for year, info in summary.items():
    all_dta_files.extend(info["dta_files"])

all_dta_files

[PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta'),
 PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta'),
 PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta')]

In [7]:
import pandas as pd

for year, info in summary.items():
    print(f"\n=== Year: {year} ===")

    if len(info["dta_files"]) == 0:
        print("[WARNING] No .dta file found")
        continue

    dta_path = info["dta_files"][0]
    df = pd.read_stata(dta_path)

    print("File:", dta_path)
    print("Shape:", df.shape)


=== Year: 2021 ===


/tmp/ipykernel_6384/1464113481.py:11: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


File: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta
Shape: (1565139, 62)

=== Year: 2022 ===
File: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta
Shape: (1545502, 62)

=== Year: 2023 ===
File: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta
Shape: (1625833, 62)


In [10]:
import pandas as pd
from pathlib import Path


def save_teds_dta_as_csv(summary, output_dir=None):
    """
    Convert extracted TEDS-A .dta files to year-specific CSV files.

    Parameters
    ----------
    summary : dict
        Output from extract_selected_teds_stata_zips()
    output_dir : str or Path or None
        Directory to save CSV files. If None, saves inside each year's extract_dir.

    Returns
    -------
    csv_summary : dict
        Saved CSV paths by year.
    """

    csv_summary = {}

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    for year, info in summary.items():
        print(f"\n========== {year} ==========")

        dta_files = info.get("dta_files", [])

        if len(dta_files) == 0:
            print(f"[WARNING] No .dta file found for {year}")
            csv_summary[year] = {
                "status": "no_dta",
                "csv_path": None
            }
            continue

        # 보통 TEDS-A zip 안에는 메인 .dta 하나가 있음
        dta_path = dta_files[0]
        print(f"[INFO] Reading DTA file:")
        print(f"       {dta_path}")

        df = pd.read_stata(dta_path)

        print(f"[INFO] Data shape: {df.shape}")

        if output_dir is None:
            save_dir = info["extract_dir"]
        else:
            save_dir = output_dir

        csv_path = save_dir / f"TEDS_A_{year}.csv"

        df.to_csv(csv_path, index=False)

        print(f"[OK] Saved CSV:")
        print(f"     {csv_path}")

        csv_summary[year] = {
            "status": "ok",
            "dta_path": dta_path,
            "csv_path": csv_path,
            "shape": df.shape
        }

    return csv_summary

In [11]:
BASE_DIR = Path(os.environ.get("NSDUH_DATA_DIR", "."))

csv_summary = save_teds_dta_as_csv(
    summary=summary,
    output_dir=BASE_DIR / "TEDS_A_2021_2023_csv"
)


========== 2021 ==========
[INFO] Reading DTA file:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta


/tmp/ipykernel_6384/3754019487.py:46: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


[INFO] Data shape: (1565139, 62)
[OK] Saved CSV:
     /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2021.csv

========== 2022 ==========
[INFO] Reading DTA file:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta


/tmp/ipykernel_6384/3754019487.py:46: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


[INFO] Data shape: (1545502, 62)
[OK] Saved CSV:
     /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2022.csv

========== 2023 ==========
[INFO] Reading DTA file:
       /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta


/tmp/ipykernel_6384/3754019487.py:46: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


[INFO] Data shape: (1625833, 62)
[OK] Saved CSV:
     /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2023.csv


In [12]:
for year, info in csv_summary.items():
    print(year, info)

2021 {'status': 'ok', 'dta_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta'), 'csv_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2021.csv'), 'shape': (1565139, 62)}
2022 {'status': 'ok', 'dta_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta'), 'csv_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2022.csv'), 'shape': (1545502, 62)}
2023 {'status': 'ok', 'dta_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta'), 'csv_path': PosixPath('/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_2023_csv/TEDS_A_2023.csv'), 'shape': (1625833, 62)}


In [13]:
for year, info in csv_summary.items():
    if info["status"] == "ok":
        df_test = pd.read_csv(info["csv_path"], nrows=5)
        print(f"\n========== {year} ==========")
        print(df_test.shape)
        display(df_test.head())


========== 2021 ==========
(5, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2021,1354749,Alaska,"13 years of college, university, or vocationa...",Now married,"Rehab/residential, long term (more than 30 days)",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
1,2021,1264433,Alaska,Grades 9 to 11,Never married,"Rehab/residential, long term (more than 30 days)",Probation/parole,No prior treatment episodes,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,Missing/unknown/not collected/invalid
2,2021,1242624,Alaska,"13 years of college, university, or vocationa...",Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,No prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Fairbanks, AK Metropolitan Statistical Area"
3,2021,1207458,Alaska,Grades 9 to 11,Now married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
4,2021,1279064,Alaska,Grades 9 to 11,Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Five or more prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"



========== 2022 ==========
(5, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2022,1328633,Alaska,Grade 12 (or GED),Separated,"Rehab/residential, short term (30 days or fewer)",Missing/unknown/not collected/invalid,One prior treatment episode,Other health care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Juneau, AK Micropolitan Statistical Area"
1,2022,1235701,Alaska,"13 years of college, university, or vocationa...",Never married,"Rehab/residential, long term (more than 30 days)",Missing/unknown/not collected/invalid,Two prior treatment episodes,Alcohol/drug use care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Anchorage, AK Metropolitan Statistical Area"
2,2022,1213309,Alaska,Grade 12 (or GED),Now married,"Rehab/residential, short term (30 days or fewer)",Missing/unknown/not collected/invalid,One prior treatment episode,Other health care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Juneau, AK Micropolitan Statistical Area"
3,2022,1177199,Alaska,Grade 12 (or GED),Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,No prior treatment episodes,Other community referral,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
4,2022,1250756,Alaska,Grade 12 (or GED),"Divorced, widowed","Ambulatory, intensive outpatient",Probation/parole,One prior treatment episode,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"



========== 2023 ==========
(5, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2023,1373091,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,Two prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
1,2023,1264567,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
2,2023,1238288,Alaska,Grade 12 (or GED),Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Anchorage, AK Metropolitan Statistical Area"
3,2023,1196178,Alaska,"Less than one school grade, no schooling, nurs...",Never married,"Rehab/residential, long term (more than 30 days)",Prison,Four prior treatment episodes,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
4,2023,1282103,Alaska,Grade 12 (or GED),"Divorced, widowed","Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Three prior treatment episodes,Individual (includes self-referral),Once,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,Missing/unknown/not collected/invalid


In [14]:
import pandas as pd
from pathlib import Path

# 저장할 폴더
OUTPUT_DIR = Path("/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_summary = {}

for year, info in summary.items():
    print(f"\n========== {year} ==========")

    dta_files = info.get("dta_files", [])

    if len(dta_files) == 0:
        print(f"[WARNING] No DTA file found for {year}")
        continue

    # 보통 각 zip 안에 메인 .dta 파일 하나가 있음
    dta_path = dta_files[0]

    print(f"[INFO] Reading: {dta_path}")

    df = pd.read_stata(dta_path)

    print("[INFO] Full dataframe shape:", df.shape)
    print("[INFO] Head shape:", df.head().shape)
    display(df.head())

    csv_path = OUTPUT_DIR / f"TEDS_A_{year}.csv"

    df.to_csv(csv_path, index=False)

    print(f"[OK] Saved CSV to: {csv_path}")

    csv_summary[year] = {
        "dta_path": dta_path,
        "csv_path": csv_path,
        "shape": df.shape,
        "columns": list(df.columns)
    }


========== 2021 ==========
[INFO] Reading: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2021_stata_extracted/tedsa_puf_2021.dta


/tmp/ipykernel_6384/3485340815.py:24: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


[INFO] Full dataframe shape: (1565139, 62)
[INFO] Head shape: (5, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2021,1354749,Alaska,"13 years of college, university, or vocationa...",Now married,"Rehab/residential, long term (more than 30 days)",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
1,2021,1264433,Alaska,Grades 9 to 11,Never married,"Rehab/residential, long term (more than 30 days)",Probation/parole,No prior treatment episodes,Court/criminal justice referral/DUI/DWI,None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,Missing/unknown/not collected/invalid
2,2021,1242624,Alaska,"13 years of college, university, or vocationa...",Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,No prior treatment episodes,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Fairbanks, AK Metropolitan Statistical Area"
3,2021,1207458,Alaska,Grades 9 to 11,Now married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
4,2021,1279064,Alaska,Grades 9 to 11,Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Five or more prior treatment episodes,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"


[OK] Saved CSV to: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2021.csv

========== 2022 ==========
[INFO] Reading: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2022_stata_extracted/tedsa_puf_2022.dta


...

[OK] Saved CSV to: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2022.csv

========== 2023 ==========
[INFO] Reading: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_2023_stata_extracted/tedsa_puf_2023.dta


/tmp/ipykernel_6384/3485340815.py:24: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(dta_path)


[INFO] Full dataframe shape: (1625833, 62)
[INFO] Head shape: (5, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2023,1373091,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,Two prior treatment episodes,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
1,2023,1264567,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
2,2023,1238288,Alaska,Grade 12 (or GED),Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Anchorage, AK Metropolitan Statistical Area"
3,2023,1196178,Alaska,"Less than one school grade, no schooling, nurs...",Never married,"Rehab/residential, long term (more than 30 days)",Prison,Four prior treatment episodes,Court/criminal justice referral/DUI/DWI,None,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
4,2023,1282103,Alaska,Grade 12 (or GED),"Divorced, widowed","Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Three prior treatment episodes,Individual (includes self-referral),Once,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,Missing/unknown/not collected/invalid


[OK] Saved CSV to: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2023.csv


In [15]:
for year, info in csv_summary.items():
    print(year)
    print("CSV:", info["csv_path"])
    print("Shape:", info["shape"])

2021
CSV: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2021.csv
Shape: (1565139, 62)
2022
CSV: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2022.csv
Shape: (1545502, 62)
2023
CSV: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2023.csv
Shape: (1625833, 62)


In [16]:
df_2021 = pd.read_csv(OUTPUT_DIR / "TEDS_A_2021.csv")
df_2022 = pd.read_csv(OUTPUT_DIR / "TEDS_A_2022.csv")
df_2023 = pd.read_csv(OUTPUT_DIR / "TEDS_A_2023.csv")

print(df_2021.shape)
print(df_2022.shape)
print(df_2023.shape)

(1565139, 62)
(1545502, 62)
(1625833, 62)


In [ ]:
dfs = {
    2021: df_2021,
    2022: df_2022,
    2023: df_2023,
}

OUTPUT_DIR = Path("/data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year")  # adjust this to your path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for year, df in dfs.items():
    print(f"\n========== {year} ==========")
    print("Full shape:", df.shape)
    display(df.head())

    csv_path = OUTPUT_DIR / f"TEDS_A_{year}.csv"
    df.to_csv(csv_path, index=False)

    print(f"[OK] Saved: {csv_path}")


========== 2021 ==========
Full shape: (1565139, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2021,1354749,Alaska,"13 years of college, university, or vocationa...",Now married,"Rehab/residential, long term (more than 30 days)",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
1,2021,1264433,Alaska,Grades 9 to 11,Never married,"Rehab/residential, long term (more than 30 days)",Probation/parole,No prior treatment episodes,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,Missing/unknown/not collected/invalid
2,2021,1242624,Alaska,"13 years of college, university, or vocationa...",Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,No prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,"Fairbanks, AK Metropolitan Statistical Area"
3,2021,1207458,Alaska,Grades 9 to 11,Now married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
4,2021,1279064,Alaska,Grades 9 to 11,Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Five or more prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"


[OK] Saved: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2021.csv

========== 2022 ==========
Full shape: (1545502, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2022,1328633,Alaska,Grade 12 (or GED),Separated,"Rehab/residential, short term (30 days or fewer)",Missing/unknown/not collected/invalid,One prior treatment episode,Other health care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Juneau, AK Micropolitan Statistical Area"
1,2022,1235701,Alaska,"13 years of college, university, or vocationa...",Never married,"Rehab/residential, long term (more than 30 days)",Missing/unknown/not collected/invalid,Two prior treatment episodes,Alcohol/drug use care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Anchorage, AK Metropolitan Statistical Area"
2,2022,1213309,Alaska,Grade 12 (or GED),Now married,"Rehab/residential, short term (30 days or fewer)",Missing/unknown/not collected/invalid,One prior treatment episode,Other health care provider,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Juneau, AK Micropolitan Statistical Area"
3,2022,1177199,Alaska,Grade 12 (or GED),Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,No prior treatment episodes,Other community referral,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
4,2022,1250756,Alaska,Grade 12 (or GED),"Divorced, widowed","Ambulatory, intensive outpatient",Probation/parole,One prior treatment episode,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"


[OK] Saved: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2022.csv

========== 2023 ==========
Full shape: (1625833, 62)


,ADMYR,CASEID,STFIPS,EDUC,MARSTAT,SERVICES,DETCRIM,NOPRIOR,PSOURCE,ARRESTS,...,BARBFLG,SEDHPFLG,INHFLG,OTCFLG,OTHERFLG,DIVISION,REGION,IDU,ALCDRUG,CBSA2020
0,2023,1373091,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,Two prior treatment episodes,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
1,2023,1264567,Alaska,Grades 9 to 11,Never married,"Detox, 24hour, free-standing residential",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance reported,Pacific,West,IDU not reported,Other drugs only,"Anchorage, AK Metropolitan Statistical Area"
2,2023,1238288,Alaska,Grade 12 (or GED),Never married,"Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,One prior treatment episode,Individual (includes self-referral),NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol only,"Anchorage, AK Metropolitan Statistical Area"
3,2023,1196178,Alaska,"Less than one school grade, no schooling, nurs...",Never married,"Rehab/residential, long term (more than 30 days)",Prison,Four prior treatment episodes,Court/criminal justice referral/DUI/DWI,NaN,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU reported,Alcohol and other drugs,"Anchorage, AK Metropolitan Statistical Area"
4,2023,1282103,Alaska,Grade 12 (or GED),"Divorced, widowed","Ambulatory, non-intensive outpatient",Missing/unknown/not collected/invalid,Three prior treatment episodes,Individual (includes self-referral),Once,...,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Substance not reported,Pacific,West,IDU not reported,Alcohol and other drugs,Missing/unknown/not collected/invalid


[OK] Saved: /data/users/jupyter-yos225/venvs/proflee/drug/TEDS_A_csv_by_year/TEDS_A_2023.csv
